# SemEval DiCo-NLI · A1 · Kaggle T4×2

在 mDeBERTa 的 B2 生成行上比较 Q0 硬重标、Q0 软标签和均匀标签平滑。
原始行全部保留金标。正式任务是 45 项，B2 未通过复用检查时为 60 项。

选择 **GPU T4 x2**，打开 Internet，Add Input 挂载 `A1_Kaggle_Train_v1.zip`。
第一次保持 `RUN_FORMAL = False`，Run All，下载 `a1_train_v1_results.zip` 看 SMOKE。
SMOKE 通过后把开关改成 `True`，用 **Save & Run All**。中断时 Add Input 挂上上次的结果包再运行。

In [ ]:
RUN_FORMAL = False

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
OUT = WORK / "a1_run"
RESULT = WORK / "a1_train_v1_results.zip"
DATASET = "A1_Kaggle_Train_v1"

dirs = sorted({p.parent for p in INPUT.rglob("run_a1.py")
               if (p.parent / "teacher/teacher_targets.jsonl").is_file()
               and (p.parent / "SHA256SUMS").is_file()})
zips = sorted(INPUT.rglob(DATASET + ".zip"))
if len(dirs) + len(zips) != 1:
    raise RuntimeError(f"请挂恰好一份 A1 数据集。解压目录 {dirs}，ZIP {zips}")

if dirs:
    BUNDLE = dirs[0]
else:
    BUNDLE = WORK / DATASET
    with zipfile.ZipFile(zips[0]) as archive:
        for member in archive.namelist():
            path = Path(member)
            if path.is_absolute() or ".." in path.parts or path.parts[0] != DATASET:
                raise ValueError(f"训练包路径不合法：{member}")
        archive.extractall(WORK)

base = [sys.executable, str(BUNDLE / "run_a1.py")]
common = ["--bundle", str(BUNDLE), "--out", str(OUT), "--result", str(RESULT)]
os.environ["TOKENIZERS_PARALLELISM"] = "false"
subprocess.run(base + ["preflight"] + common, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.57.6", "sentencepiece>=0.2", "protobuf>=4.25", "scikit-learn>=1.3"],
               check=True)

In [ ]:
from IPython.display import FileLink, display

def run_stage(stage, extra=(), env=None):
    try:
        subprocess.run(base + [stage] + common + list(extra), env=env, check=True)
    finally:
        if RESULT.is_file():
            display(FileLink(str(RESULT.relative_to(WORK))))

run_stage("smoke", env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})

In [ ]:
if RUN_FORMAL:
    resumes = sorted(INPUT.rglob(RESULT.name))
    resumes += sorted(p.parent for p in INPUT.rglob("run_identity.json")
                      if p.parent.name == OUT.name and (p.parent / "tasks").is_dir())
    if len(resumes) > 1:
        raise RuntimeError(f"请只挂一份旧结果，找到 {resumes}")
    extra = ["--resume", str(resumes[0])] if resumes else []
    run_stage("full", extra)
else:
    print("本次只跑 SMOKE。检查结果后再打开 RUN_FORMAL。")